In [12]:
import os
import glob
import pandas as pd
import numpy as np
import zipfile
from scipy.fft import fft, fftfreq
from scipy.signal import medfilt
import matplotlib.pyplot as plt

# ============================================================
# 設定與路徑
# ============================================================
base_path = r"E:\EarthScienceFair_Data"
target_folders = ["5"]
WINDOW_SIZE = 150
STRIDE = 5

# ============================================================
# 工具函式
# ============================================================

def clean_for_window(raw_data):
    s = pd.to_numeric(pd.Series(raw_data), errors='coerce')
    if s.isna().all(): return np.array([])
    s = s.interpolate(method='linear', limit_direction='both').ffill().bfill()
    val = s.values
    return val - np.mean(val) if len(val) > 0 else np.array([])


def get_sliding_freq(time, signal, win_size, stride):
    t_res, f_res = [], []
    for start in range(0, len(signal) - win_size, stride):
        end = start + win_size
        seg_t = time[start:end]
        seg_s = signal[start:end]

        N = len(seg_s)
        dt = np.mean(np.diff(seg_t))
        if dt <= 0: continue

        yf = fft(seg_s)
        xf = fftfreq(N, dt)[:N//2]
        amp = 2.0/N * np.abs(yf[:N//2])

        mask = (xf >= 1.2) & (xf <= 5.0)
        if np.any(mask) and np.sum(amp[mask]) > 0:
            f_main = np.sum(xf[mask] * amp[mask]) / np.sum(amp[mask])
            t_res.append(np.mean(seg_t))
            f_res.append(f_main)
    return np.array(t_res), np.array(f_res)

# ============================================================
# 主流程
# ============================================================
xlsx_files = []
for folder in target_folders:
    fp = os.path.join(base_path, folder)
    if os.path.exists(fp):
        found = glob.glob(os.path.join(fp, "**", "*.xlsx"), recursive=True)
        print(f"資料夾 {fp} 找到 {len(found)} 個 xlsx 檔案")
        xlsx_files.extend(found)
    else:
        print(f"資料夾不存在: {fp}")

for tracker_file in xlsx_files:
    file_id = os.path.splitext(os.path.basename(tracker_file))[0]
    print(f"\n處理中: {file_id}")

    try:
        ACC_THRESHOLD_CSV  = 0.5   # csv 用較高門檻（max=8.19，靜止約0.15）
        ACC_THRESHOLD_XLSX = 0.14  # xlsx ya 範圍 0.11~0.17，取中間偏高

        # ── 1. 先讀 xlsx ──────────────────────────────────────
        df = pd.read_excel(tracker_file, header=2)
        df_clean = df[pd.to_numeric(df['t_s'], errors='coerce').notna()].copy().reset_index(drop=True)
        print(f"  xlsx 有效行數: {len(df_clean)}/{len(df)} 筆")

        t_s_raw = pd.to_numeric(df_clean['t_s'], errors='coerce').values

        # xlsx 觸發點
        ya_abs = np.abs(pd.to_numeric(df_clean['ya'], errors='coerce').values)
        trigger_xlsx = np.argmax(ya_abs > ACC_THRESHOLD_XLSX)
        print(f"  xlsx 觸發點: index={trigger_xlsx}, t={t_s_raw[trigger_xlsx]:.4f}")
        print(f"  xlsx ya 絕對值: min={ya_abs.min():.4f}, max={ya_abs.max():.4f}, 中位數={np.median(ya_abs):.4f}")

        t_s = t_s_raw[trigger_xlsx:] - t_s_raw[trigger_xlsx]

        target_layers = {
            "ya": df_clean['ya'].values[trigger_xlsx:] if 'ya' in df_clean.columns else None,
            "yd": df_clean['yd'].values[trigger_xlsx:] if 'yd' in df_clean.columns else None,
            "yh": df_clean['yh'].values[trigger_xlsx:] if 'yh' in df_clean.columns else None,
            "yi": df_clean['yi'].values[trigger_xlsx:] if 'yi' in df_clean.columns else None,
        }

        plt.figure(figsize=(12, 6))

        # ── 2. 再讀 csv ───────────────────────────────────────
        zip_path = os.path.join(os.path.dirname(tracker_file), f"{file_id}.zip")
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, 'r') as z:
                csv_files = [f for f in z.namelist() if f.endswith('.csv')]
                if csv_files:
                    with z.open(csv_files[0]) as f:
                        acc_df = pd.read_csv(f)

                    # 先定義原始時間軸和觸發點
                    t_acc_raw = pd.to_numeric(acc_df.iloc[:, 0], errors='coerce').values
                    acc_abs = np.abs(pd.to_numeric(acc_df.iloc[:, -1], errors='coerce').values)
                    print(f"  csv acc 絕對值: min={acc_abs.min():.4f}, max={acc_abs.max():.4f}, 中位數={np.median(acc_abs):.4f}")

                    trigger_acc = np.argmax(acc_abs > ACC_THRESHOLD_CSV)
                    print(f"  csv 觸發點: index={trigger_acc}, t={t_acc_raw[trigger_acc]:.4f}")

                    # 再裁切觸發點之後的資料
                    t_acc = t_acc_raw[trigger_acc:] - t_acc_raw[trigger_acc]
                    v_acc_raw = acc_df.iloc[:, -1].values[trigger_acc:]

                    # 最後裁切到 xlsx 時間範圍
                    t_max_xlsx = t_s[-1]
                    csv_mask = t_acc <= t_max_xlsx
                    t_acc = t_acc[csv_mask]
                    v_acc = clean_for_window(v_acc_raw[csv_mask])
                    print(f"  csv 裁切後: {len(t_acc)} 筆, 時間範圍 0 ~ {t_acc[-1]:.2f} s")

                    dt_acc  = np.median(np.diff(t_acc[t_acc > 0][:100])) if len(t_acc) > 10 else 0.01
                    dt_xlsx = np.median(np.diff(t_s[:100]))
                    win_acc    = max(int(WINDOW_SIZE * dt_xlsx / dt_acc), 30)
                    stride_acc = max(int(STRIDE     * dt_xlsx / dt_acc),  5)
                    t_p, f_p = get_sliding_freq(t_acc, v_acc, win_acc, stride_acc)
                    if len(t_p) > 0:
                        plt.plot(t_p, f_p, 'k--', label="Input (Accel-CSV)", alpha=0.8, linewidth=2)
                    print(f"  CSV: {len(t_acc)} 筆, win={win_acc}, stride={stride_acc}")
        else:
            print(f"  zip 不存在，略過 CSV")

        # ── 3. 畫 xlsx 層位 ───────────────────────────────────
        colors = {"ya": "red", "yd": "orange", "yh": "green", "yi": "blue"}
        for name, raw_val in target_layers.items():
            if raw_val is None: continue
            clean_sig = clean_for_window(raw_val)
            if len(clean_sig) > WINDOW_SIZE:
                sig_med = medfilt(clean_sig, kernel_size=5)
                tp, fp = get_sliding_freq(t_s, sig_med, WINDOW_SIZE, STRIDE)
                if len(tp) > 0:
                    print(f"  層位 {name}: 頻率範圍 = {fp.min():.3f} ~ {fp.max():.3f} Hz")
                    plt.plot(tp, fp, color=colors[name], label=f"Layer {name}", alpha=0.7)

        plt.title(f"Sliding Window Analysis: {file_id}")
        plt.xlabel("Time (s)")
        plt.ylabel("Frequency (Hz)")
        plt.grid(True, alpha=0.3)
        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.savefig(f"Sliding_{file_id}.png", dpi=150)
        plt.close()

        dt_acc  = np.median(np.diff(t_acc[:100]))
        dt_xlsx = np.median(np.diff(t_s[:100]))
        window_sec_acc  = win_acc * dt_acc
        window_sec_xlsx = WINDOW_SIZE * dt_xlsx

        print(f"  xlsx 觸發後: {len(t_s)} 筆, 時間範圍 0 ~ {t_s[-1]:.2f} s")
        print(f"  csv  觸發後: {len(t_acc)} 筆, 時間範圍 0 ~ {t_acc[-1]:.2f} s")
        print(f"  dt_xlsx={dt_xlsx:.4f}s, dt_acc={dt_acc:.4f}s")
        print(f"  xlsx 窗口={WINDOW_SIZE}點 = {window_sec_xlsx:.2f}s")
        print(f"  csv  窗口={win_acc}點 = {window_sec_acc:.2f}s")

    except Exception as e:
        print(f"失敗 {file_id}: {e}")

資料夾 E:\EarthScienceFair_Data\5 找到 22 個 xlsx 檔案

處理中: 5-G1-1
  xlsx 有效行數: 470/1695 筆
  xlsx 觸發點: index=164, t=5.4667
  xlsx ya 絕對值: min=0.0601, max=0.1416, 中位數=0.1043
  csv acc 絕對值: min=0.0224, max=8.1912, 中位數=0.0955
  csv 觸發點: index=693, t=6.9583
  csv 裁切後: 1014 筆, 時間範圍 0 ~ 10.17 s
  CSV: 1014 筆, win=498, stride=16
  層位 ya: 頻率範圍 = 2.031 ~ 2.756 Hz
  層位 yd: 頻率範圍 = 1.842 ~ 2.478 Hz
  層位 yi: 頻率範圍 = 1.854 ~ 2.419 Hz
  xlsx 觸發後: 306 筆, 時間範圍 0 ~ 10.17 s
  csv  觸發後: 1014 筆, 時間範圍 0 ~ 10.17 s
  dt_xlsx=0.0333s, dt_acc=0.0100s
  xlsx 窗口=150點 = 5.00s
  csv  窗口=498點 = 5.00s

處理中: 5-G1-2
  xlsx 有效行數: 524/1934 筆
  xlsx 觸發點: index=17, t=0.5667
  xlsx ya 絕對值: min=0.0808, max=0.1641, 中位數=0.1302
  csv acc 絕對值: min=0.0272, max=8.1146, 中位數=0.5789
  csv 觸發點: index=288, t=2.8934
  csv 裁切後: 1681 筆, 時間範圍 0 ~ 16.86 s
  CSV: 1681 筆, win=498, stride=16
  層位 ya: 頻率範圍 = 2.010 ~ 2.923 Hz
  層位 yd: 頻率範圍 = 1.885 ~ 2.670 Hz
  層位 yi: 頻率範圍 = 1.883 ~ 2.625 Hz
  xlsx 觸發後: 507 筆, 時間範圍 0 ~ 16.87 s
  csv  觸發後: 1681 筆, 時間範圍 0 